#### Description: real application for U.S. energy generation compositions and New York City Citi bike sharing system
#### generate required TimePFN model results needed for Figures S5 and S11.

In [ ]:
import sys
import os


if os.path.exists('../TimePFN'):
    repo_path = os.path.abspath('../TimePFN')
elif os.path.exists('../TimePFN-main'):
    repo_path = os.path.abspath('../TimePFN-main')
else:
    raise FileNotFoundError("🚨 ERROR: Could not find the TimePFN folder! Are you sure it's located one folder up from this notebook?")

print(f"Found the repository at: {repo_path}")


if repo_path not in sys.path:
    sys.path.insert(0, repo_path)


try:
    from model.TimePFN import Model as TimePFN_Model
    print("✅ Model imported successfully!")
except Exception as e:
    print(f"🚨 Import failed! Error: {e}")

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import copy
import random
import matplotlib.pyplot as plt
from scipy.special import rel_entr
import torch.nn.functional as F


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



class GeodesicLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super(GeodesicLoss, self).__init__()
        self.eps = eps

    def forward(self, predictions, targets):
        # Calculate dot product (cosine similarity for normalized vectors)
        cos_sim = F.cosine_similarity(predictions, targets)
        
        # Clamp to avoid NaN gradients at exactly 1.0 or -1.0
        cos_sim = torch.clamp(cos_sim, -1.0 + self.eps, 1.0 - self.eps)
        
        # Calculate the actual angle (arc length on a unit sphere)
        return torch.acos(cos_sim).mean()

class TimePFNForecaster:
    def __init__(self, model, seq_length=20, max_epochs=8, lr=1e-4, weight_decay=1e-4):
        self.seq_length = seq_length
        self.max_epochs = max_epochs
        self.lr = lr
        self.weight_decay = weight_decay
        self.model = model.to(device)
        self.criterion = nn.MSELoss() 
        ##self.criterion = GeodesicLoss()
        self.history = {'train_loss': [], 'val_loss': []}
        
    def create_sequences(self, X_data, Y_data):
        X_seq, y_seq = [], []
        for i in range(len(X_data) - self.seq_length):
            X_seq.append(X_data[i : i + self.seq_length])
            y_seq.append(Y_data[i + self.seq_length])
        return torch.tensor(np.array(X_seq), dtype=torch.float32), torch.tensor(np.array(y_seq), dtype=torch.float32)

    def fit(self, X_window, Y_window, val_ratio=0.2, batch_size=16, verbose=False):
        X_all, y_all = self.create_sequences(X_window, Y_window)
        split_idx = int(len(X_all) * (1 - val_ratio))
        
        X_train, y_train = X_all[:split_idx].to(device), y_all[:split_idx].to(device)
        X_val, y_val = X_all[split_idx:].to(device), y_all[split_idx:].to(device)
        
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        optimizer = optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        best_val_loss, best_weights = float('inf'), None
        self.history = {'train_loss': [], 'val_loss': []}
        
        for epoch in range(self.max_epochs):
            self.model.train()
            epoch_train_loss = 0.0
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                
                # --- THE FIX: Pass None for the 3 missing temporal/decoder arguments ---
                outputs = self.model(batch_x, None, None, None)
                
                pred = outputs[:, -1, :] if outputs.dim() == 3 else outputs
                loss = self.criterion(pred, batch_y)
                loss.backward()
                optimizer.step()
                epoch_train_loss += loss.item() * batch_x.size(0)
            
            avg_train_loss = epoch_train_loss / len(train_loader.dataset)
            
            self.model.eval()
            with torch.no_grad():
                # --- THE FIX: Pass None here too ---
                val_outputs = self.model(X_val, None, None, None)
                
                val_pred = val_outputs[:, -1, :] if val_outputs.dim() == 3 else val_outputs
                val_loss = self.criterion(val_pred, y_val).item()
                
            self.history['train_loss'].append(avg_train_loss)
            self.history['val_loss'].append(val_loss)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_weights = copy.deepcopy(self.model.state_dict())
                
        if best_weights is not None: self.model.load_state_dict(best_weights)

    def predict(self, initial_x_seq, steps_ahead, future_exo=None):
        self.model.eval()
        curr_seq = torch.tensor(initial_x_seq, dtype=torch.float32).unsqueeze(0).to(device)
        preds = []
        with torch.no_grad():
            for i in range(steps_ahead):
                # --- THE FIX: Pass None for autoregressive steps ---
                outputs = self.model(curr_seq, None, None, None)
                
                next_y = outputs[:, -1, :] if outputs.dim() == 3 else outputs
                
                # Enforce Spherical Manifold
                next_y = next_y / torch.norm(next_y, p=2, dim=1, keepdim=True)
                preds.append(next_y.cpu().numpy()[0])
                
                if future_exo is not None:
                    exo = torch.tensor(future_exo[i], dtype=torch.float32).to(device)
                    next_x = torch.cat((next_y[0], exo), dim=0).unsqueeze(0).unsqueeze(0)
                else:
                    next_x = next_y.unsqueeze(1)
                curr_seq = torch.cat((curr_seq[:, 1:, :], next_x), dim=1)
        return np.array(preds)


def calculate_spherical_errors(predictions, targets):
    y_pred = predictions / np.linalg.norm(predictions, axis=1, keepdims=True)
    y_true = targets / np.linalg.norm(targets, axis=1, keepdims=True)
    dot_products = np.clip(np.sum(y_pred * y_true, axis=1), -1.0, 1.0)
    return np.mean(np.arccos(dot_products))

def rolling_window_cv(X, Y, kappa, forecaster, verbose=False):
    T, target_dim = Y.shape[0], Y.shape[1]
    train_size = int(np.floor(T * kappa))
    has_exo = X.shape[1] > target_dim
    results = []
    
    max_m = T - train_size
    print(f"Total windows to process: {max_m}")
    print("-" * 70)
    
    for m in range(max_m, 0, -1):
        train_start, train_end = T - train_size - m, T - m
        X_train, Y_train = X[train_start:train_end], Y[train_start:train_end]
        Y_test = Y[train_end : train_end + m]
        future_exo = X[train_end : train_end + m, target_dim:] if has_exo else None
        
        forecaster.fit(X_train, Y_train, val_ratio=0.2, verbose=verbose)
        
        final_train_loss = forecaster.history['train_loss'][-1] if forecaster.history['train_loss'] else 0
        final_val_loss = forecaster.history['val_loss'][-1] if forecaster.history['val_loss'] else 0
        
        seed_seq = X_train[-forecaster.seq_length:]
        Y_pred = forecaster.predict(seed_seq, steps_ahead=m, future_exo=future_exo)
        
        error = calculate_spherical_errors(Y_pred, Y_test)
        results.append({"m": m, "Sphere_Divergence": error, "Final_Train_Loss": final_train_loss, "Final_Val_Loss": final_val_loss})
        
        print(f"Completed m={m:02d} | Train: {final_train_loss:.6f} | Val: {final_val_loss:.6f} | Sphere Error: {error:.6f}")
            
    return pd.DataFrame(results)

def calculate_js_divergence(predictions, targets):
    """
    Calculates the average Jensen-Shannon Divergence.
    Original and predicted data are squared first to form probability distributions.
    """
    # 1. Enforce strict L2 normalization (just in case model output drifted)
    y_pred = predictions / np.linalg.norm(predictions, axis=1, keepdims=True)
    y_true = targets / np.linalg.norm(targets, axis=1, keepdims=True)
    
    # 2. Square the data (Maps spherical coordinates to probability distributions)
    P = np.square(y_true)
    Q = np.square(y_pred)
    
    # Add a microscopic epsilon to prevent log(0) errors in relative entropy
    epsilon = 1e-10
    P = np.clip(P, epsilon, 1.0)
    Q = np.clip(Q, epsilon, 1.0)
    
    # Re-normalize to ensure they sum to exactly 1.0 after clipping
    P = P / np.sum(P, axis=1, keepdims=True)
    Q = Q / np.sum(Q, axis=1, keepdims=True)
    
    # 3. Calculate Jensen-Shannon Divergence
    # M is the midpoint distribution
    M = 0.5 * (P + Q)
    
    # rel_entr(x, y) computes x * log(x / y). We sum across the features (axis=1).
    kl_pm = np.sum(rel_entr(P, M), axis=1)
    kl_qm = np.sum(rel_entr(Q, M), axis=1)
    
    js_divergences = 0.5 * kl_pm + 0.5 * kl_qm
    
    # 4. Return the average JS Divergence over the forecast horizon m
    return np.mean(js_divergences)

def rolling_window_cv_JSD(X, Y, kappa, forecaster, verbose=False):
    """
    Rolling window cross-validation tracking Jensen-Shannon Divergence.
    """
    T, target_dim = Y.shape[0], Y.shape[1]
    train_size = int(np.floor(T * kappa))
    has_exo = X.shape[1] > target_dim
    results = []
    
    max_m = T - train_size
    print(f"Total windows to process (JSD): {max_m}")
    print("-" * 70)
    
    for m in range(max_m, 0, -1):
        train_start, train_end = T - train_size - m, T - m
        X_train, Y_train = X[train_start:train_end], Y[train_start:train_end]
        Y_test = Y[train_end : train_end + m]
        future_exo = X[train_end : train_end + m, target_dim:] if has_exo else None
        
        # Fit the forecaster
        forecaster.fit(X_train, Y_train, val_ratio=0.2, verbose=verbose)
        
        # Extract losses safely (in case max_epochs=0 for zero-shot)
        final_train_loss = forecaster.history['train_loss'][-1] if forecaster.history.get('train_loss') else 0
        final_val_loss = forecaster.history['val_loss'][-1] if forecaster.history.get('val_loss') else 0
        
        # Predict
        seed_seq = X_train[-forecaster.seq_length:]
        Y_pred = forecaster.predict(seed_seq, steps_ahead=m, future_exo=future_exo)
        
        # Calculate JSD
        jsd_error = calculate_js_divergence(Y_pred, Y_test)
        
        # Store results matching the old format
        results.append({
            "m": m, 
            "JSD_Divergence": jsd_error, 
            "Final_Train_Loss": final_train_loss, 
            "Final_Val_Loss": final_val_loss
        })
        
        print(f"Completed m={m:02d} | Train: {final_train_loss:.6f} | Val: {final_val_loss:.6f} | JSD: {jsd_error:.6f}")
            
    return pd.DataFrame(results)

 ### Real data - U.S. energy generation composition

In [ ]:
print("\n--- Preparing Data ---")
df = pd.read_csv('energy.csv', header=None)
Y_raw = df.values

# Enforce Spherical Manifold constraint
Y_target = Y_raw / np.linalg.norm(Y_raw, axis=1, keepdims=True)
T, target_dim = Y_target.shape

X_base = Y_target.copy()
base_input_dim = X_base.shape[1]

print(f"Target Data (Y) Shape: {Y_target.shape}")
kappa = 0.892
seq_len = 96

if os.path.exists('../TimePFN'):
    repo_path = os.path.abspath('../TimePFN')
elif os.path.exists('../TimePFN-main'):
    repo_path = os.path.abspath('../TimePFN-main')
else:
    raise FileNotFoundError("🚨 ERROR: Could not find the TimePFN folder! Are you sure it's located one folder up from this notebook?")

print(f"Found the repository at: {repo_path}")


if repo_path not in sys.path:
    sys.path.insert(0, repo_path)


try:
    from model.TimePFN import Model as TimePFN_Model
    print("✅ Model imported successfully!")
except Exception as e:
    print(f"🚨 Import failed! Error: {e}")

class Configs:
    def __init__(self):
        # --- Reverse-Engineered from the .pth file! ---
        self.seq_len = 96            
        self.pred_len = 96           
        
        # The split architecture!
        self.embed_dim = 256         # Transformer internal dimension
        self.d_model = 1024          # Final projection head dimension
        self.d_ff = 512              # Feed-forward network dimension
        
        self.enc_in = base_input_dim
        self.c_out = target_dim
        
        self.use_norm = True         
        self.patch_size = 16         
        self.class_strategy = 'projection' 
        self.factor = 3              
        self.activation = 'gelu'     
        
        self.n_heads = 8
        self.e_layers = 3
        self.dropout = 0.1
        self.stride = 8
        self.output_attention = False

configs = Configs()
raw_timepfn = TimePFN_Model(configs).to(device)

weights_path = os.path.join(repo_path, 'checkpoints', 'TimePFN', 'checkpoint.pth')
if os.path.exists(weights_path):
    state_dict = torch.load(weights_path, map_location=device)
    raw_timepfn.load_state_dict(state_dict, strict=False) 
    print("✅ Pre-trained weights loaded successfully!")
else:
    print(f"⚠️ Warning: Weights missing at '{weights_path}'. Training from scratch.")


timepfn_forecaster = TimePFNForecaster(model=raw_timepfn, seq_length=seq_len, max_epochs=8, lr=1e-4)

print("\n--- Running TimePFN Rolling Window CV ---")
df_timepfn = rolling_window_cv(X_base, Y_target, kappa, timepfn_forecaster, verbose=False)

# Clean up results
df_results = df_timepfn[['m', 'Sphere_Divergence']].copy()
df_results.columns = ['Forecast Horizon (m)', 'TimePFN Error (rad)']
df_results = df_results.sort_values('Forecast Horizon (m)')

# Save CSV
df_results.to_csv("timepfn_energy_results.csv", index=False)
print("\n=== TimePFN Benchmark Results Saved ===")
print(df_results.to_string(index=False, float_format="%.6f"))


plt.figure(figsize=(10, 6))
plt.plot(df_results['Forecast Horizon (m)'], df_results['TimePFN Error (rad)'], 
         marker='^', color='#2ca02c', linestyle='-', linewidth=2, label="TimePFN (Pre-Trained)")

plt.title(r'TimePFN Prediction Error ${\rm Dist}(m)$', fontsize=14)
plt.xlabel('Forecast Horizon (m)', fontsize=12)
plt.ylabel(r'Mean Geodesic Distance (Radians)', fontsize=12)
plt.xticks(df_results['Forecast Horizon (m)'])
plt.gca().invert_xaxis() 
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Real data - New York City Citi bike sharing system

In [ ]:
print("\n--- Preparing Data ---")
df = pd.read_csv('trip.csv', header=None)
Y_raw = df.values

# Enforce Spherical Manifold constraint
Y_target = Y_raw / np.linalg.norm(Y_raw, axis=1, keepdims=True)
T, target_dim = Y_target.shape

X_base = Y_target.copy()
base_input_dim = X_base.shape[1]

print(f"Target Data (Y) Shape: {Y_target.shape}")
kappa =  0.86
seq_len = 96

if os.path.exists('../TimePFN'):
    repo_path = os.path.abspath('../TimePFN')
elif os.path.exists('../TimePFN-main'):
    repo_path = os.path.abspath('../TimePFN-main')
else:
    raise FileNotFoundError("🚨 ERROR: Could not find the TimePFN folder! Are you sure it's located one folder up from this notebook?")

print(f"Found the repository at: {repo_path}")


if repo_path not in sys.path:
    sys.path.insert(0, repo_path)


try:
    from model.TimePFN import Model as TimePFN_Model
    print("✅ Model imported successfully!")
except Exception as e:
    print(f"🚨 Import failed! Error: {e}")

class Configs:
    def __init__(self):
        # --- Reverse-Engineered from the .pth file ---
        self.seq_len = 96            
        self.pred_len = 96           
        
        # The split architecture
        self.embed_dim = 256         # Transformer internal dimension
        self.d_model = 1024          # Final projection head dimension
        self.d_ff = 512              # Feed-forward network dimension
        
        self.enc_in = base_input_dim
        self.c_out = target_dim
        
        self.use_norm = True         
        self.patch_size = 16         
        self.class_strategy = 'projection' 
        self.factor = 3              
        self.activation = 'gelu'     
        
        self.n_heads = 8
        self.e_layers = 3
        self.dropout = 0.1
        self.stride = 8
        self.output_attention = False

configs = Configs()
raw_timepfn = TimePFN_Model(configs).to(device)

weights_path = os.path.join(repo_path, 'checkpoints', 'TimePFN', 'checkpoint.pth')
if os.path.exists(weights_path):
    state_dict = torch.load(weights_path, map_location=device)
    raw_timepfn.load_state_dict(state_dict, strict=False) 
    print("✅ Pre-trained weights loaded successfully!")
else:
    print(f"⚠️ Warning: Weights missing at '{weights_path}'. Training from scratch.")


timepfn_forecaster = TimePFNForecaster(model=raw_timepfn, seq_length=seq_len, max_epochs=8, lr=1e-4)

print("\n--- Running TimePFN Rolling Window CV ---")
df_timepfn = rolling_window_cv_JSD(X_base, Y_target, kappa, timepfn_forecaster)

# Clean up results
df_results = df_timepfn[['m', 'JSD_Divergence']].copy()
df_results.columns = ['Forecast Horizon (m)', 'TimePFN Error (rad)']
df_results = df_results.sort_values('Forecast Horizon (m)')

# Save CSV
df_results.to_csv("timepfn_trip_results.csv", index=False)
print("\n=== TimePFN Benchmark Results Saved ===")
print(df_results.to_string(index=False, float_format="%.6f"))


plt.figure(figsize=(10, 6))
plt.plot(df_results['Forecast Horizon (m)'], df_results['TimePFN Error (rad)'], 
         marker='^', color='#2ca02c', linestyle='-', linewidth=2, label="TimePFN (Pre-Trained)")

plt.title(r'TimePFN Prediction Error ${\rm Dist}(m)$', fontsize=14)
plt.xlabel('Forecast Horizon (m)', fontsize=12)
plt.ylabel(r'Mean Geodesic Distance (Radians)', fontsize=12)
plt.xticks(df_results['Forecast Horizon (m)'])
plt.gca().invert_xaxis() 
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()